# 02 — The Event Loop: One Loop to Rule Them All

**Goal:** Understand what the event loop is, how tasks are scheduled, and what blocks it.

## What is the event loop?

It's a `while True` loop that:
1. Picks the next ready task
2. Runs it until it hits an `await`
3. Puts it back in the waiting list
4. Picks the next ready task
5. Repeat forever

```
Event Loop:
  ready_queue: [task_A, task_B, task_C]
  waiting_list: [task_D (waiting for network), task_E (waiting for timer)]
  
  while True:
      task = ready_queue.pop()
      run task until it hits 'await'
      if task is waiting for I/O → move to waiting_list
      if I/O is complete for any waiting task → move to ready_queue
```

There is only ONE event loop per thread. Everything runs inside it.

## Exercise 2.1 — Seeing the event loop in action

Let's add print statements to see exactly when tasks run.

In [ ]:
import asyncio
import time

async def task(name, steps):
    for i in range(steps):
        print(f"  [{time.time():.3f}] {name}: running step {i+1}/{steps}")
        await asyncio.sleep(0)  # yield to event loop
    print(f"  [{time.time():.3f}] {name}: DONE")

print("Starting 3 tasks...")
await asyncio.gather(
    task("A", 3),
    task("B", 2),
    task("C", 4),
)

### Question 2.1
Look at the output. Do A, B, C take turns? Who goes first after each yield? Is it round-robin or something else?

*Your answer:*



## Exercise 2.2 — BLOCKING the event loop

What happens if a task does something slow WITHOUT `await`?

In [ ]:
async def greedy_task():
    print(f"[{time.time():.1f}] Greedy: I'm going to hog the loop for 3 seconds...")
    time.sleep(3)  # BLOCKING — no await, event loop is FROZEN
    print(f"[{time.time():.1f}] Greedy: done")

async def polite_task():
    print(f"[{time.time():.1f}] Polite: I want to run!")
    await asyncio.sleep(0.1)
    print(f"[{time.time():.1f}] Polite: finally got a chance!")

await asyncio.gather(greedy_task(), polite_task())

### Question 2.2
When did polite_task get to print "I want to run"? Before or after greedy_task's 3-second sleep? Why?

*Your answer:*



## Exercise 2.3 — The fix: run_in_executor

When you MUST call blocking code (a library that doesn't support async), wrap it in `run_in_executor`. This runs it in a separate thread so the event loop stays free.

In [ ]:
def blocking_function():
    """Pretend this is a library that ONLY has sync API."""
    time.sleep(2)
    return "result from blocking code"

async def smart_task():
    print(f"[{time.time():.1f}] Smart: calling blocking code in executor...")
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(None, blocking_function)
    print(f"[{time.time():.1f}] Smart: got {result}")

async def other_task():
    print(f"[{time.time():.1f}] Other: I can run while blocking code runs!")
    await asyncio.sleep(0.5)
    print(f"[{time.time():.1f}] Other: see? I wasn't blocked!")

await asyncio.gather(smart_task(), other_task())

### Question 2.3
How does `run_in_executor` work? Where does the blocking code actually run? Why doesn't it freeze the event loop?

*Your answer:*



## The three rules

```
1. Between two awaits, YOU are the only one running. Nobody can interrupt you.
2. If you block without await, EVERYONE else freezes.
3. Use run_in_executor for blocking code you can't change.
```

**Next notebook:** asyncio.Queue — passing messages between tasks